In [3]:
import random

train_data = [
    {"A": i * 10 % 100, "B": i * 7 % 100, "C": (c := random.randint(0, 100)), "sum": i * 10 % 100 + i * 7 % 100 + c}
    for i in range(100)
]

eval_data = [
    {"A": i * 10 % 100, "B": i * 7 % 100, "C": (c := random.randint(0, 100)), "sum": i * 10 % 100 + i * 7 % 100 + c}
    for i in range(80, 120)
]

train_data[:10]

[{'A': 0, 'B': 0, 'C': 40, 'sum': 40},
 {'A': 10, 'B': 7, 'C': 37, 'sum': 54},
 {'A': 20, 'B': 14, 'C': 28, 'sum': 62},
 {'A': 30, 'B': 21, 'C': 30, 'sum': 81},
 {'A': 40, 'B': 28, 'C': 88, 'sum': 156},
 {'A': 50, 'B': 35, 'C': 34, 'sum': 119},
 {'A': 60, 'B': 42, 'C': 60, 'sum': 162},
 {'A': 70, 'B': 49, 'C': 26, 'sum': 145},
 {'A': 80, 'B': 56, 'C': 47, 'sum': 183},
 {'A': 90, 'B': 63, 'C': 58, 'sum': 211}]

In [8]:
from origami import ModelConfig, OrigamiConfig, OrigamiPipeline, TrainingConfig
from origami.config import DataConfig
from origami.training import TableLogCallback, accuracy

config = OrigamiConfig(
    data=DataConfig(
        infer_schema=True,
        numeric_mode="discretize",
        cat_threshold=5,
        n_bins=5,
    ),
    model=ModelConfig(
        backbone="transformer",
        kvpe_pooling="sum",
        d_model=32,
        n_heads=4,
        n_layers=6,
        d_ff=256,
        dropout=0.0,
    ),
    training=TrainingConfig(
        shuffle_keys=False,
        batch_size=16,
        warmup_steps=100,
        learning_rate=1e-3,
        eval_strategy="epoch",
        eval_steps=100,
        eval_metrics={"acc": accuracy},
        eval_sample_size=100,
        eval_on_train=True,
        target_key="sum",
        target_loss_weight=1.0,
        constrain_grammar=True,
        constrain_schema=True,
    ),
    device="mps",
)

pipeline = OrigamiPipeline(config)
pipeline.fit(
    train_data,
    eval_data=eval_data,
    callbacks=[TableLogCallback(print_every=10)],
    epochs=500,
    verbose=True,
)


Discretized fields (4):
  - A: 5 bins
  - B: 5 bins
  - C: 5 bins
  - sum: 5 bins
Vocabulary size: 34
Derived schema:
{
  "type": "object",
  "properties": {
    "A": {
      "type": "integer",
      "enum": [
        0,
        10,
        20,
        30,
        40,
        "... + 5 more"
      ],
      "minimum": 0,
      "maximum": 90
    },
    "B": {
      "type": "integer",
      "enum": [
        0,
        1,
        2,
        3,
        4,
        "... + 95 more"
      ],
      "minimum": 0,
      "maximum": 99
    },
    "C": {
      "type": "integer",
      "enum": [
        1,
        2,
        4,
        5,
        6,
        "... + 57 more"
      ],
      "minimum": 1,
      "maximum": 100
    },
    "sum": {
      "type": "integer",
      "enum": [
        28,
        36,
        40,
        41,
        52,
        "... + 78 more"
      ],
      "minimum": 28,
      "maximum": 256
    }
  },
  "additionalProperties": false,
  "required": [
    "A",
    "B",
    "C",
 

OrigamiPipeline(numeric_mode='discretize', fitted)

In [9]:
gen = pipeline.generate(1000)

import pandas as pd

df = pd.DataFrame(gen)
print(f"Unique A: {df['A'].unique()}")
print(f"Unique B: {df['B'].unique()}")
print(f"Unique C: {df['C'].unique()}")

Unique A: [10 20 80 40 60]
Unique B: [10 29 69 89 49]
Unique C: [62 84  9 27 46]
